In [1]:
import os
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

try:
    from scipy.special import eval_genlaguerre, factorial
except Exception:
    eval_genlaguerre = None
    factorial = math.factorial

# ==========================================================
#  Configuración general
# ==========================================================
SEED = 1234
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64)

# Constantes físicas
ALPHA = 1.0 / 137.036
MEC2_EV = 510_998.95000  # eV
Z = 1.0

# Guardar directamente en Colab
OUT_DIR = "/content"
os.makedirs(OUT_DIR, exist_ok=True)

# ==========================================================
#  Estados a entrenar
# ==========================================================
STATES = [
    dict(label=r"$2s_{1/2}$", n=2, l=0, j=0.5, kappa=-1, epochs=8000),
    dict(label=r"$2p_{1/2}$", n=2, l=1, j=0.5, kappa=+1, epochs=8000),
    dict(label=r"$2p_{3/2}$", n=2, l=1, j=1.5, kappa=-2, epochs=8000),
    dict(label=r"$3p_{3/2}$", n=3, l=1, j=1.5, kappa=-2, epochs=8000),
]

# Pesos de la pérdida
LAMBDA_ODE = 1.0
LAMBDA_BC = 10.0
LAMBDA_NORM = 1.0
LAMBDA_E = 0.05

# Épocas que se guardan para comparar R_PINN(r) con R_teórica(r)
SNAPSHOT_EPOCHS = [1000, 4000, 8000]

# ==========================================================
#  Física: eta, energía de referencia y radial teórica
# ==========================================================
def eta_ducharme(kappa: int) -> float:
    """
    eta = |kappa| - sqrt(kappa^2 - alpha^2)
    """
    return abs(kappa) - math.sqrt(kappa**2 - ALPHA**2)


def epsilon_dirac(n: int, kappa: int) -> float:
    """
    epsilon = E_total/(m_e c^2)
    """
    sqrt_term = math.sqrt(kappa**2 - ALPHA**2)
    denom = n - abs(kappa) + sqrt_term
    return (1.0 + ALPHA**2 / denom**2) ** (-0.5)


def binding_ev_from_epsilon(epsilon):
    """
    Energía de ligadura en eV.
    Valor negativo cercano a -13.6/n^2 eV.
    """
    return (epsilon - 1.0) * MEC2_EV


def hydrogenic_radial_np(r: np.ndarray, n: int, l: int) -> np.ndarray:
    """
    Función radial hidrogenoide no relativista R_{n,l}(r),
    usada como referencia teórica visual.

    Corrección importante:
    - 2s_{1/2}  usa R_{2,0}
    - 2p_{1/2}  usa R_{2,1}
    - 2p_{3/2}  usa R_{2,1}
    - 3p_{3/2}  usa R_{3,1}
    """
    rho = 2.0 * Z * r / n

    if eval_genlaguerre is None:
        # Fórmulas explícitas en unidades de a0

        if n == 2 and l == 0:
            # R_{2,0}
            R = (1.0 / (2.0 * math.sqrt(2.0))) * (2.0 - r) * np.exp(-r / 2.0)

        elif n == 2 and l == 1:
            # R_{2,1}
            # Esta es la curva teórica correcta para 2p_{1/2} y 2p_{3/2}
            R = (1.0 / (2.0 * math.sqrt(6.0))) * r * np.exp(-r / 2.0)

        elif n == 3 and l == 1:
            # R_{3,1}
            R = (4.0 / (81.0 * math.sqrt(6.0))) * r * (6.0 - r) * np.exp(-r / 3.0)

        else:
            R = (r**l) * np.exp(-r / n)

    else:
        pref = math.sqrt(
            (2.0 * Z / n)**3
            * factorial(n - l - 1)
            / (2.0 * n * factorial(n + l))
        )

        Lag = eval_genlaguerre(n - l - 1, 2 * l + 1, rho)
        R = pref * np.exp(-rho / 2.0) * rho**l * Lag

    norm = np.trapezoid(R**2 * r**2, r)
    return R / math.sqrt(max(norm, 1e-30))


def normalize_radial_np(R: np.ndarray, r: np.ndarray) -> np.ndarray:
    norm = np.trapezoid(R**2 * r**2, r)
    return R / math.sqrt(max(norm, 1e-30))


def align_to_theory(R: np.ndarray, R_ref: np.ndarray, r: np.ndarray) -> np.ndarray:
    """
    Alinea signo y escala de la PINN respecto a la solución teórica.
    """
    Rn = normalize_radial_np(R, r)
    overlap = np.trapezoid(Rn * R_ref * r**2, r)

    if overlap < 0:
        Rn = -Rn

    return Rn

# ==========================================================
#  Red neuronal PINN
# ==========================================================
class RadialPINN(nn.Module):
    def __init__(self, n: int, l: int, r_max: float, width: int = 64, depth: int = 4):
        super().__init__()

        self.n = n
        self.l = l
        self.r_max = r_max

        layers = []
        layers.append(nn.Linear(1, width))
        layers.append(nn.Tanh())

        for _ in range(depth - 1):
            layers.append(nn.Linear(width, width))
            layers.append(nn.Tanh())

        layers.append(nn.Linear(width, 1))

        self.net = nn.Sequential(*layers)

        # beta controla la cola exponencial.
        self.log_beta = nn.Parameter(
            torch.tensor(math.log(1.0 / n), dtype=torch.float64)
        )

    def forward(self, r):
        x = 2.0 * r / self.r_max - 1.0
        raw = self.net(x)
        beta = torch.exp(self.log_beta)

        # Parametrización física:
        # comportamiento cerca del origen y decaimiento asintótico.
        return (r ** self.l) * torch.exp(-beta * r) * raw


def residual_kg(model, r, epsilon, eta):
    """
    Residual radial KG-Ducharme en r/a0.
    """
    r.requires_grad_(True)

    R = model(r)

    dR = torch.autograd.grad(
        R,
        r,
        grad_outputs=torch.ones_like(R),
        create_graph=True
    )[0]

    d2R = torch.autograd.grad(
        dR,
        r,
        grad_outputs=torch.ones_like(dR),
        create_graph=True
    )[0]

    coeff = (
        (epsilon**2 - 1.0) / ALPHA**2
        + 2.0 * Z * epsilon / r
        + eta * (1.0 - eta) / r**2
    )

    res = d2R + (2.0 / r) * dR + coeff * R

    return res, R

# ==========================================================
#  Entrenamiento por estado
# ==========================================================
def train_state(state):
    label = state["label"]
    n = state["n"]
    l = state["l"]
    kappa = state["kappa"]
    epochs = state["epochs"]

    eta = eta_ducharme(kappa)

    eps_ref = epsilon_dirac(n, kappa)
    e_ref_ev = binding_ev_from_epsilon(eps_ref)

    r_min = 1e-3
    r_max = 35.0 * n

    n_col = 320
    n_norm = 600

    model = RadialPINN(
        n=n,
        l=l,
        r_max=r_max,
        width=64,
        depth=4
    ).to(device)

    # Parámetro entrenable de energía total normalizada:
    # epsilon = E/(m_e c^2)
    eps_raw = nn.Parameter(
        torch.tensor(0.0, dtype=torch.float64, device=device)
    )

    eps_window = 2.0e-5

    params = list(model.parameters()) + [eps_raw]

    optimizer = torch.optim.Adam(params, lr=1.0e-3)

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=2500,
        gamma=0.55
    )

    r_norm = torch.linspace(
        r_min,
        r_max,
        n_norm,
        device=device
    ).reshape(-1, 1)

    r_plot_np = np.linspace(
        r_min,
        min(r_max, 35.0),
        700
    )

    r_plot = torch.tensor(
        r_plot_np.reshape(-1, 1),
        device=device,
        dtype=torch.float64
    )

    R_ref_np = hydrogenic_radial_np(r_plot_np, n, l)

    history = {
        "epoch": [],
        "loss_total": [],
        "loss_ode": [],
        "loss_bc": [],
        "loss_norm": [],
        "loss_E": [],
        "energy_ev": [],
    }

    snapshots = {}

    rng = np.random.default_rng(SEED + 37 * n + 5 * abs(kappa))

    print("\n" + "═" * 72)
    print(f"Entrenando {label} | n={n}, l={l}, kappa={kappa:+d}")
    print(f"E_teórica = {e_ref_ev:.8f} eV | epsilon_ref = {eps_ref:.12f}")
    print("═" * 72)

    for ep in range(1, epochs + 1):

        optimizer.zero_grad()

        # Muestreo más denso cerca del núcleo
        u = rng.random(n_col)
        r_col_np = r_min + (r_max - r_min) * (u**1.7)

        r_col = torch.tensor(
            r_col_np.reshape(-1, 1),
            device=device,
            dtype=torch.float64
        )

        epsilon = eps_ref + eps_window * torch.tanh(eps_raw)

        res, _ = residual_kg(model, r_col, epsilon, eta)

        loss_ode = torch.mean(res**2)

        # Condiciones de frontera
        R_left = model(
            torch.tensor([[r_min]], device=device, dtype=torch.float64)
        )

        R_right = model(
            torch.tensor([[r_max]], device=device, dtype=torch.float64)
        )

        loss_bc = R_left.pow(2).mean() + R_right.pow(2).mean()

        # Normalización radial: integral R² r² dr = 1
        Rn = model(r_norm)

        integral = torch.trapezoid(
            (Rn[:, 0]**2) * (r_norm[:, 0]**2),
            r_norm[:, 0]
        )

        loss_norm = (integral - 1.0)**2

        # Energía en eV
        energy_ev = binding_ev_from_epsilon(epsilon)

        # Guía débil de energía
        loss_E = ((energy_ev - e_ref_ev) / 10.0)**2

        loss_total = (
            LAMBDA_ODE * loss_ode
            + LAMBDA_BC * loss_bc
            + LAMBDA_NORM * loss_norm
            + LAMBDA_E * loss_E
        )

        loss_total.backward()

        torch.nn.utils.clip_grad_norm_(
            params,
            max_norm=50.0
        )

        optimizer.step()
        scheduler.step()

        if ep == 1 or ep % 25 == 0 or ep in SNAPSHOT_EPOCHS:
            history["epoch"].append(ep)
            history["loss_total"].append(float(loss_total.detach().cpu()))
            history["loss_ode"].append(float(loss_ode.detach().cpu()))
            history["loss_bc"].append(float((LAMBDA_BC * loss_bc).detach().cpu()))
            history["loss_norm"].append(float((LAMBDA_NORM * loss_norm).detach().cpu()))
            history["loss_E"].append(float((LAMBDA_E * loss_E).detach().cpu()))
            history["energy_ev"].append(float(energy_ev.detach().cpu()))

        if ep in SNAPSHOT_EPOCHS or ep == epochs:
            with torch.no_grad():
                R_pred = model(r_plot).detach().cpu().numpy().reshape(-1)

            snapshots[ep] = align_to_theory(
                R_pred,
                R_ref_np,
                r_plot_np
            )

        if ep == 1 or ep % 1000 == 0:
            print(
                f"ep {ep:5d}/{epochs} | "
                f"L={float(loss_total.detach().cpu()):.3e} | "
                f"L_ODE={float(loss_ode.detach().cpu()):.3e} | "
                f"E_PINN={float(energy_ev.detach().cpu()):.8f} eV"
            )

    # Resultado final
    with torch.no_grad():
        epsilon_final = eps_ref + eps_window * torch.tanh(eps_raw)

        e_final_ev = float(
            binding_ev_from_epsilon(epsilon_final).detach().cpu()
        )

        R_final = model(r_plot).detach().cpu().numpy().reshape(-1)

    snapshots[epochs] = align_to_theory(
        R_final,
        R_ref_np,
        r_plot_np
    )

    return {
        "label": label,
        "n": n,
        "l": l,
        "j": state["j"],
        "kappa": kappa,
        "eta": eta,
        "epochs": epochs,
        "eps_ref": eps_ref,
        "E_ref_ev": e_ref_ev,
        "E_final_ev": e_final_ev,
        "abs_error_ev": abs(e_final_ev - e_ref_ev),
        "r_plot": r_plot_np,
        "R_ref": R_ref_np,
        "history": history,
        "snapshots": snapshots,
    }

# ==========================================================
#  Estilo de gráficas
# ==========================================================
def set_paper_style():
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],
        "font.size": 11,
        "axes.linewidth": 1.0,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "legend.fontsize": 9,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "figure.dpi": 150,
        "savefig.dpi": 300,
    })


def clean_state_name(label):
    """
    Convierte etiquetas LaTeX en nombres limpios para archivos.
    """
    name = label.replace("$", "")
    name = name.replace("\\", "")
    name = name.replace("{", "")
    name = name.replace("}", "")
    name = name.replace("_", "")
    name = name.replace("/", "")
    name = name.replace(" ", "")
    return name


def radial_theory_label(res):
    """
    Etiqueta correcta para la función radial teórica.
    """
    n = res["n"]
    l = res["l"]

    if n == 2 and l == 0:
        return r"Teórica $R_{2,0}$"

    elif n == 2 and l == 1:
        return r"Teórica $R_{2,1}$"

    elif n == 3 and l == 1:
        return r"Teórica $R_{3,1}$"

    else:
        return rf"Teórica $R_{{{n},{l}}}$"

# ==========================================================
#  Gráficas de pérdidas
# ==========================================================
def plot_losses(results):
    """
    Grafica las pérdidas de todos los estados y además una figura separada
    para cada estado.
    """
    nstates = len(results)

    # Figura general con todos los estados
    fig, axes = plt.subplots(
        1,
        nstates,
        figsize=(6.8 * nstates, 4.8),
        squeeze=False
    )

    axes = axes.ravel()

    for ax, res in zip(axes, results):
        h = res["history"]
        ep = h["epoch"]

        ax.semilogy(ep, h["loss_total"], lw=2.4, label=r"$\mathcal{L}_{total}$")
        ax.semilogy(ep, h["loss_ode"], lw=1.6, label=r"$\mathcal{L}_{ODE}$")
        ax.semilogy(ep, h["loss_bc"], lw=1.6, label=r"$\lambda_{BC}\mathcal{L}_{BC}$")
        ax.semilogy(ep, h["loss_norm"], lw=1.6, label=r"$\lambda_{norm}\mathcal{L}_{norm}$")

        if LAMBDA_E > 0:
            ax.semilogy(ep, h["loss_E"], lw=1.4, label=r"$\lambda_E\mathcal{L}_E$")

        ax.set_title(f"Loss vs epoch — {res['label']}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend(frameon=False)

    fig.tight_layout()

    path = os.path.join(OUT_DIR, "loss_vs_epoch_all_states.png")
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)

    print(f"Guardado: {path}")

    # Figuras separadas por estado
    for res in results:
        h = res["history"]
        ep = h["epoch"]

        fig, ax = plt.subplots(figsize=(7.2, 5.0))

        ax.semilogy(ep, h["loss_total"], lw=2.5, label=r"$\mathcal{L}_{total}$")
        ax.semilogy(ep, h["loss_ode"], lw=1.8, label=r"$\mathcal{L}_{ODE}$")
        ax.semilogy(ep, h["loss_bc"], lw=1.8, label=r"$\lambda_{BC}\mathcal{L}_{BC}$")
        ax.semilogy(ep, h["loss_norm"], lw=1.8, label=r"$\lambda_{norm}\mathcal{L}_{norm}$")

        if LAMBDA_E > 0:
            ax.semilogy(ep, h["loss_E"], lw=1.5, label=r"$\lambda_E\mathcal{L}_E$")

        ax.set_title(f"Loss components — {res['label']}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend(frameon=False)

        fig.tight_layout()

        filename = f"loss_components_{clean_state_name(res['label'])}.png"
        path = os.path.join(OUT_DIR, filename)

        fig.savefig(path, bbox_inches="tight")
        plt.close(fig)

        print(f"Guardado: {path}")

# ==========================================================
#  Gráficas radiales
# ==========================================================
def plot_radial_snapshots(results):
    """
    Grafica las soluciones radiales PINN comparadas con la función radial teórica.
    También guarda una figura separada para cada estado.
    """
    nstates = len(results)

    # Figura general con todos los estados
    fig, axes = plt.subplots(
        1,
        nstates,
        figsize=(6.8 * nstates, 4.8),
        squeeze=False
    )

    axes = axes.ravel()

    for ax, res in zip(axes, results):
        r = res["r_plot"]

        ax.plot(
            r,
            res["R_ref"],
            color="black",
            lw=2.6,
            label=radial_theory_label(res)
        )

        ordered_epochs = sorted(res["snapshots"].keys())

        for ep in ordered_epochs:
            lw = 2.4 if ep == res["epochs"] else 1.4
            alpha = 0.95 if ep == res["epochs"] else 0.75

            ax.plot(
                r,
                res["snapshots"][ep],
                lw=lw,
                alpha=alpha,
                label=f"PINN epoch {ep}"
            )

        ax.set_title(rf"Radial solution — {res['label']}")
        ax.set_xlabel(r"$r/a_0$")
        ax.set_ylabel(r"$R(r)$")
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False)

    fig.tight_layout()

    path = os.path.join(OUT_DIR, "radial_snapshots_all_states.png")
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)

    print(f"Guardado: {path}")

    # Figuras separadas por estado
    for res in results:
        r = res["r_plot"]

        fig, ax = plt.subplots(figsize=(7.2, 5.0))

        ax.plot(
            r,
            res["R_ref"],
            color="black",
            lw=2.8,
            label=radial_theory_label(res)
        )

        ordered_epochs = sorted(res["snapshots"].keys())

        for ep in ordered_epochs:
            lw = 2.5 if ep == res["epochs"] else 1.5
            alpha = 0.95 if ep == res["epochs"] else 0.75

            ax.plot(
                r,
                res["snapshots"][ep],
                lw=lw,
                alpha=alpha,
                label=f"PINN epoch {ep}"
            )

        ax.set_title(rf"Radial approximation — {res['label']}")
        ax.set_xlabel(r"$r/a_0$")
        ax.set_ylabel(r"$R(r)$")
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False)

        fig.tight_layout()

        filename = f"radial_approx_{clean_state_name(res['label'])}.png"
        path = os.path.join(OUT_DIR, filename)

        fig.savefig(path, bbox_inches="tight")
        plt.close(fig)

        print(f"Guardado: {path}")

# ==========================================================
#  Gráficas de energía separadas
# ==========================================================
def plot_energy_ev(results):
    """
    Genera una figura independiente por cada estado.
    Hace zoom automático en el eje Y para ver las fluctuaciones.
    Guarda cada figura en /content.
    """
    for res in results:
        h = res["history"]

        epochs = np.array(h["epoch"], dtype=float)
        energy = np.array(h["energy_ev"], dtype=float)
        e_ref = float(res["E_ref_ev"])

        fig, ax = plt.subplots(figsize=(7.5, 5.0))

        ax.plot(
            epochs,
            energy,
            lw=2.0,
            label=rf"PINN {res['label']}"
        )

        ax.axhline(
            e_ref,
            ls="--",
            lw=1.5,
            alpha=0.85,
            label=rf"Referencia {res['label']}"
        )

        ax.scatter(
            epochs[-1],
            energy[-1],
            s=45,
            zorder=5,
            label=rf"E final = {energy[-1]:.8f} eV"
        )

        # Zoom automático en el eje Y
        y_min = min(np.min(energy), e_ref)
        y_max = max(np.max(energy), e_ref)
        delta = y_max - y_min

        if delta < 1e-9:
            margin = 1e-5
        else:
            margin = 0.35 * delta

        ax.set_ylim(y_min - margin, y_max + margin)

        ax.set_title(rf"Energy fluctuation — {res['label']}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Binding energy (eV)")
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False)

        fig.tight_layout()

        filename = f"energy_zoom_{clean_state_name(res['label'])}.png"
        path = os.path.join(OUT_DIR, filename)

        fig.savefig(path, bbox_inches="tight")
        plt.close(fig)

        print(f"Guardado: {path}")

# ==========================================================
#  Tabla resumen
# ==========================================================
def plot_summary_table(results):
    fig, ax = plt.subplots(figsize=(9.5, 2.8))

    ax.axis("off")

    rows = []

    for res in results:
        rows.append([
            res["label"].replace("$", ""),
            f"{res['E_final_ev']:.8f}",
            f"{res['E_ref_ev']:.8f}",
            f"{res['abs_error_ev']:.3e}",
        ])

    table = ax.table(
        cellText=rows,
        colLabels=[
            "State",
            "E_PINN (eV)",
            "E_theory (eV)",
            "|error| (eV)"
        ],
        loc="center",
        cellLoc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.6)

    ax.set_title("PINN energy summary", pad=14)

    path = os.path.join(OUT_DIR, "summary_table_ev.png")

    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)

    print(f"Guardado: {path}")

# ==========================================================
#  Ejecución principal
# ==========================================================
def main():
    set_paper_style()

    print("\nPINN KG-Ducharme — entrenamiento radial")
    print(f"Device: {device}")
    print("Estados:")

    for s in STATES:
        eps_ref = epsilon_dirac(s["n"], s["kappa"])

        print(
            f"  {s['label']}: "
            f"n={s['n']}, "
            f"l={s['l']}, "
            f"kappa={s['kappa']:+d}, "
            f"E_ref={binding_ev_from_epsilon(eps_ref):.8f} eV"
        )

    results = [train_state(s) for s in STATES]

    plot_losses(results)
    plot_radial_snapshots(results)
    plot_energy_ev(results)
    plot_summary_table(results)

    print("\nResumen final")
    print("State        E_PINN (eV)       E_theory (eV)      |error| (eV)")
    print("-" * 66)

    for r in results:
        print(
            f"{r['label']:12s} "
            f"{r['E_final_ev']:16.8f} "
            f"{r['E_ref_ev']:16.8f} "
            f"{r['abs_error_ev']:14.3e}"
        )

    print("\nArchivos PNG guardados en /content:")

    for f in sorted(os.listdir(OUT_DIR)):
        if f.endswith(".png"):
            print(os.path.join(OUT_DIR, f))


if __name__ == "__main__":
    main()


PINN KG-Ducharme — entrenamiento radial
Device: cpu
Estados:
  $2s_{1/2}$: n=2, l=0, kappa=-1, E_ref=-3.40147984 eV
  $2p_{1/2}$: n=2, l=1, kappa=+1, E_ref=-3.40147984 eV
  $2p_{3/2}$: n=2, l=1, kappa=-2, E_ref=-3.40143456 eV
  $3p_{3/2}$: n=3, l=1, kappa=-2, E_ref=-1.51175037 eV

════════════════════════════════════════════════════════════════════════
Entrenando $2s_{1/2}$ | n=2, l=0, kappa=-1
E_teórica = -3.40147984 eV | epsilon_ref = 0.999993343470
════════════════════════════════════════════════════════════════════════
ep     1/8000 | L=8.836e+00 | L_ODE=7.731e+00 | E_PINN=-3.40147984 eV
ep  1000/8000 | L=1.216e+00 | L_ODE=2.157e-01 | E_PINN=-3.40985884 eV
ep  2000/8000 | L=1.004e+00 | L_ODE=3.683e-03 | E_PINN=-3.40170238 eV
ep  3000/8000 | L=1.000e+00 | L_ODE=1.260e-10 | E_PINN=-3.40147984 eV
ep  4000/8000 | L=1.000e+00 | L_ODE=2.243e-12 | E_PINN=-3.40147984 eV
ep  5000/8000 | L=1.000e+00 | L_ODE=7.797e-12 | E_PINN=-3.40147984 eV
ep  6000/8000 | L=1.000e+00 | L_ODE=5.527e-13 | E_